# TradeLab ML Module Example

This notebook demonstrates how to use the newly added ML module in `trade_lab.ml`.

Workflow:
1. Download OHLCV data.
2. Build signal/indicator-based features.
3. Create ML targets.
4. Train a neural model with `MLTrainer`.
5. Run walk-forward validation.
6. Use the trained model in `MLStrategy` + `BacktestEngine`.

In [10]:
from pathlib import Path
import sys

import pandas as pd
import yfinance as yf

# Allow running from repository root or from examples/.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from trade_lab.signals import OHLC
from trade_lab.indicators import EMA, RSI
from trade_lab.ml import (
    FutureReturn,
    FeatureScaler,
    MLTrainer,
    dense_model,
)
from trade_lab.strategies import MLStrategy
from trade_lab.backtesting.engine import BacktestEngine

> If this import fails because TensorFlow/Keras is not installed, install optional ML deps first (e.g. `pip install -e .[ml]`).

In [11]:
if MLTrainer is None:
    raise RuntimeError("MLTrainer is unavailable. Install ML dependencies: pip install -e .[ml]")

In [12]:
# 1) Download data
ticker = "SPY"
start = "2020-01-01"
end = "2026-01-01"

df = yf.download(ticker, start=start, end=end)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel("Ticker")

df = df.dropna().copy()
print(f"Downloaded {len(df)} rows for {ticker}.")
df.tail()

[*********************100%***********************]  1 of 1 completed

Downloaded 1508 rows for SPY.


Price,Close,High,Low,Open,Volume
Date,,,,,
2025-12-24,690.380005,690.830017,687.799988,687.950012,39445600
2025-12-26,690.309998,691.659973,689.270020,690.640015,41613300
2025-12-29,687.849976,689.200012,686.070007,687.539978,62559500
2025-12-30,687.010010,688.559998,686.580017,687.450012,47160700
2025-12-31,681.919983,687.359985,681.710022,687.140015,74144800


## Build Features + Target

In [13]:
# 2) Define feature pipeline (signals + indicators)
# OHLC signal is passed into EMA as an example of signal->indicator chaining.
ohlc_signal = OHLC()
indicators = [
    EMA(ohlc_signal, period=20),
    EMA(period=50),
    RSI(period=14),
]

# 3) Target: future return over 5 bars, squashed to [-1, 1]
target = FutureReturn(periods=5, column="Close", scale=10.0)

# 4) Model factory + scaler
model_builder = dense_model(layers=[64, 32], dropout=0.2, learning_rate=0.001)
scaler = FeatureScaler(method="standard")

trainer = MLTrainer(
    indicators=indicators,
    target=target,
    model_builder=model_builder,
    scaler=scaler,
)

In [14]:
X, y, feature_columns, clean_index = trainer.build_dataset(df.copy())
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("First 10 feature columns:")
feature_columns[:10]

Feature matrix shape: (1489, 7)
Target shape: (1489,)
First 10 feature columns:


['signal__log_return_open',
 'signal__log_return_high',
 'signal__log_return_low',
 'signal__log_return_close',
 'indicator__ema_20',
 'indicator__ema_50',
 'indicator__rsi_14']

## Train Model

In [15]:
trained = trainer.train(
    df=df.copy(),
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

print("Trained model ready for MLStrategy.")
print("Number of model input features:", len(trained.input_names))
trained.input_names[:10]

Epoch 1/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0965 - val_loss: 0.0584
Epoch 2/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0795 - val_loss: 0.0491
Epoch 3/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0652 - val_loss: 0.0487
Epoch 4/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0651 - val_loss: 0.0473
Epoch 5/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0610 - val_loss: 0.0459
Epoch 6/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0600 - val_loss: 0.0443
Epoch 7/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0575 - val_loss: 0.0452
Epoch 8/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0599 - val_loss: 0.0466
Epoch 9/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0580 - val_loss: 0.0445
Epoch 10/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0565 - val_loss: 0.0440
Epoch 11/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0563 - val_loss: 0.0454
Epoch 12/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0

['signal__log_return_open',
 'signal__log_return_high',
 'signal__log_return_low',
 'signal__log_return_close',
 'indicator__ema_20',
 'indicator__ema_50',
 'indicator__rsi_14']

## Walk-Forward Validation

In [16]:
wf_results = trainer.walk_forward(
    df=df.copy(),
    n_splits=3,
    expanding=True,
    initial_train_ratio=0.6,
    epochs=10,
    batch_size=32,
    verbose=0,
)

summary = pd.DataFrame({
    "fold": [r.fold for r in wf_results],
    "train_loss": [r.train_loss for r in wf_results],
    "n_test": [len(r.test_predictions) for r in wf_results],
})
summary

,fold,train_loss,n_test
0,0,0.060279,198
1,1,0.060335,198
2,2,0.055439,200


## Use Trained Model in Backtesting

In [17]:
ml_strategy = MLStrategy(
    model=trained,
    indicators=indicators,
    allow_long=True,
    allow_short=True,
    entry_threshold=0.2,
    exit_threshold=0.05,
)

engine = BacktestEngine(
    strategy=ml_strategy,
    ticker=ticker,
    start="2024-01-01",
    end="2026-01-01",
    initial_capital=100_000,
    commission=0.001,
    slippage=0.0005,
)

result = engine.run()
result.metrics

[*********************100%***********************]  1 of 1 completed


{'total_return': np.float64(0.43687783627122523),
 'annualized_return': np.float64(0.19956423360183062),
 'sharpe_ratio': np.float64(1.2247074395699526),
 'sortino_ratio': np.float64(1.535160618346866),
 'max_drawdown': np.float64(-0.18755240821113947),
 'annual_volatility': np.float64(0.1629484945987641),
 'total_trades': 1,
 'win_rate': 1.0,
 'profit_factor': inf,
 'avg_win': np.float64(43687.78362712249),
 'avg_loss': 0.0,
 'avg_trade_bars': np.float64(487.0),
 'total_commission': np.float64(243.7317151424648),
 'long_win_rate': 1.0,
 'short_win_rate': 0.0,
 'long_avg_win': np.float64(43687.78362712249),
 'long_avg_loss': 0.0,
 'short_avg_win': 0.0,
 'short_avg_loss': 0.0}

In [18]:
result.trade_log.tail()

,direction,entry_date,entry_price,exit_date,exit_price,units,pnl,return_pct,commission,bars_held
0,long,2024-01-23,473.399484,2025-12-31,681.579023,211.027057,43687.783627,0.437315,243.731715,487
